# 2 · ProteinMPNN / SolubleMPNN / fine-tuned (AlkSecMPNN, AcidSecMPNN) — sequence design
Same dauparas CLI for every variant; the **Choose variant** cell toggles soluble weights or loads a
fine-tuned checkpoint via `--path_to_model_weights`. `v_48_020` for the stock models, `v_48_002`-derived
for the fine-tunes (and a `base_v48_002` matched-base option). Temperature-only sampling. **Runtime → GPU.**


In [ ]:
#@title Step 0 — Upload & unzip the design bundle
#@markdown Upload **design_bundle.zip** (contains `design_common.py`,
#@markdown `design_input_proteins.csv`, and `structures/`).
#@markdown Build it locally with `design/make_bundle.sh`.
import os, zipfile
from google.colab import files

if not os.path.exists("design_common.py"):
    print("Upload design_bundle.zip:")
    up = files.upload()
    zname = next(iter(up))
    with zipfile.ZipFile(zname) as z:
        z.extractall(".")
    # if it unzipped into a 'design/' subdir, hoist contents to CWD
    if os.path.exists("design/design_common.py") and not os.path.exists("design_common.py"):
        import shutil
        for item in os.listdir("design"):
            shutil.move(os.path.join("design", item), item)
print("Bundle ready:", sorted(os.listdir(".")))

In [ ]:
#@title Install ProteinMPNN (clone official repo)
import os, subprocess, sys
if not os.path.isdir("ProteinMPNN"):
    subprocess.run(["git","clone","--quiet",
        "https://github.com/dauparas/ProteinMPNN.git"], check=True)
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (set GPU runtime!)")

In [ ]:
#@title Step 1 — Import shared config and show the LOCKED settings
import design_common as dc
proteins = dc.load_inputs()           # 25 templates, structure paths resolved
print(f"Loaded {len(proteins)} design templates")
print("\n=== LOCKED CONFIG (identical across all model notebooks) ===")
import dataclasses, json
cfg = {k: v for k, v in dataclasses.asdict(dc.CONFIG).items() if k != "deviations"}
print(json.dumps(cfg, indent=2, default=str))
display(proteins[["uniprot_id","species","domain","rank_class","sequence_length"]])

In [ ]:
#@title Choose variant — ProteinMPNN · SolubleMPNN · fine-tuned (AlkSecMPNN / AcidSecMPNN)
VARIANT = "ProteinMPNN"  #@param ["ProteinMPNN", "SolubleMPNN", "base_v48_002", "AlkSecMPNN", "AcidSecMPNN", "AlkSecMPNN_020", "AcidSecMPNN_020"]
#@markdown `base_v48_002` = the vanilla checkpoint the fine-tunes started from (matched base for the FT designs).
#@markdown The fine-tuned variants need **ft_mpnn_weights.zip** (from finetune/colab/) — you'll be prompted to upload it.

# (display name, soluble flag, --model_name, weights_dir for --path_to_model_weights)
_VARIANTS = {
    "ProteinMPNN":    ("ProteinMPNN",      False, "v_48_020", ""),
    "SolubleMPNN":    ("SolubleMPNN",      True,  "v_48_020", ""),
    "base_v48_002":   ("ProteinMPNN_v002", False, "v_48_002", ""),     # FT's matched base
    "AlkSecMPNN":   ("AlkSecMPNN",     False, "AlkSecMPNN",   "/content/ft_weights"),
    "AcidSecMPNN": ("AcidSecMPNN",   False, "AcidSecMPNN", "/content/ft_weights"),
    "AlkSecMPNN_020":   ("AlkSecMPNN_020",   False, "AlkSecMPNN_020",   "/content/ft_weights"),  # fine-tuned from v_48_020
    "AcidSecMPNN_020": ("AcidSecMPNN_020", False, "AcidSecMPNN_020", "/content/ft_weights"),  # fine-tuned from v_48_020
}
MODEL, SOLUBLE, MPNN_MODEL_NAME, WEIGHTS_DIR = _VARIANTS[VARIANT]

if WEIGHTS_DIR:   # fine-tuned variant: make sure the weights bundle is present
    import os, zipfile, torch
    if not (os.path.isdir(WEIGHTS_DIR) and any(f.endswith(".pt") for f in os.listdir(WEIGHTS_DIR))):
        from google.colab import files
        print("Upload finetune/colab/ft_mpnn_weights.zip:")
        up = files.upload()
        with zipfile.ZipFile(next(iter(up))) as z:
            z.extractall("/content")
    for f in os.listdir(WEIGHTS_DIR):   # CLI reads checkpoint["noise_level"]; add if missing
        if f.endswith(".pt"):
            fp = os.path.join(WEIGHTS_DIR, f); ck = torch.load(fp, map_location="cpu")
            if "noise_level" not in ck:
                ck["noise_level"] = 0.2; torch.save(ck, fp)

if WEIGHTS_DIR:
    _ckpt = os.path.join(WEIGHTS_DIR, MPNN_MODEL_NAME + ".pt")
    assert os.path.exists(_ckpt), f"FT checkpoint not found: {_ckpt} — re-upload ft_mpnn_weights.zip (Runtime may have reset)."
    print("  FT checkpoint OK:", _ckpt)

print("Variant:", MODEL, "| --model_name:", MPNN_MODEL_NAME, "| weights:", WEIGHTS_DIR or "(repo vanilla_model_weights)")


## Comparability notes — ProteinMPNN

These are the points where ProteinMPNN touches the locked settings. Anything that
**deviates** is recorded via `dc.CONFIG.note_deviation(...)` so it lands in the
output manifest.


- **Temperature**: passed via `--sampling_temp`; honoured exactly.
- **Sampler**: ProteinMPNN is temperature-only (no top-k/top-p) — matches the locked scheme.
- **Seeds**: ProteinMPNN takes one `--seed` per run; we loop over `dc.CONFIG.seeds`
  (one sequence per seed) so each sample is independently reproducible.
- **score_type** = `mpnn_global_score` (mean negative log-likelihood the model reports per sequence).

In [ ]:
#@title helper — run ProteinMPNN on one PDB for one seed, parse the FASTA
import subprocess, sys, os, re
from pathlib import Path

def run_mpnn_one(pdb_path, seed, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    cmd = [sys.executable, "ProteinMPNN/protein_mpnn_run.py",
           "--pdb_path", pdb_path,
           "--pdb_path_chains", dc.CONFIG.design_chain,
           "--out_folder", out_dir,
           "--num_seq_per_target", "1",
           "--sampling_temp", str(dc.CONFIG.temperature),
           "--seed", str(seed),
           "--batch_size", "1",
           "--model_name", MPNN_MODEL_NAME]
    if SOLUBLE:
        cmd.append("--use_soluble_model")
    if WEIGHTS_DIR:                       # fine-tuned / non-default weights
        cmd += ["--path_to_model_weights", WEIGHTS_DIR]
    try:
        subprocess.run(cmd, check=True, capture_output=True, text=True)
    except subprocess.CalledProcessError as e:
        print("\n!!! ProteinMPNN failed for", MPNN_MODEL_NAME, "seed", seed)
        print("CMD:", " ".join(cmd))
        print("STDERR:\n", (e.stderr or "")[-2500:])
        raise
    # output FASTA: out_dir/seqs/<pdbstem>.fa  (first record = WT, rest = designs)
    stem = Path(pdb_path).stem
    fa = Path(out_dir) / "seqs" / f"{stem}.fa"
    recs = _parse_fasta(fa)
    return recs

def _parse_fasta(fa):
    recs = []
    header, seq = None, []
    for line in Path(fa).read_text().splitlines():
        if line.startswith(">"):
            if header is not None:
                recs.append((header, "".join(seq)))
            header, seq = line[1:], []
        else:
            seq.append(line.strip())
    if header is not None:
        recs.append((header, "".join(seq)))
    return recs   # [(header, seq), ...]; index 0 is the native input

def _global_score(header):
    m = re.search(r"global_score=([-\d.]+)", header)
    return float(m.group(1)) if m else None

In [ ]:
#@title Step 2 — Smoke test on the shortest protein (one seed)
_p = proteins.sort_values("sequence_length").iloc[0]
_recs = run_mpnn_one(_p.structure_path, seed=dc.CONFIG.seeds[0], out_dir="outputs/_smoke")
_wt_parsed, _des = _recs[0][1], _recs[1][1]
print(f"{_p.uniprot_id} len={_p.sequence_length}")
print("DES:", _des[:60])
assert len(_des) == _p.sequence_length, f"length {len(_des)} != {_p.sequence_length}"
assert set(_des) <= set(dc.CANONICAL_AA)
print("✓ smoke test OK")

In [ ]:
#@title Step 3 — Design all 25 proteins (loop seeds = samples)
from tqdm.auto import tqdm
rows = []
for p in tqdm(list(proteins.itertuples()), desc="ProteinMPNN"):
    for i, seed in enumerate(dc.CONFIG.seeds):
        recs = run_mpnn_one(p.structure_path, seed=seed,
                            out_dir=f"outputs/mpnn/{p.uniprot_id}_s{seed}")
        header, seq = recs[1]   # first design
        rows.append(dc.make_record(p, model=MODEL, sample_idx=i, seed=seed,
                                   designed_sequence=seq,
                                   model_score=_global_score(header),
                                   score_type="mpnn_global_score",
                                   soluble_variant=SOLUBLE))
print(f"Generated {len(rows)} sequences")

In [ ]:
#@title Step 4 — Validate (faithful + comparable) and write outputs
df = dc.finalize(rows, model=MODEL, strict=True)   # raises if a guard fails
dc.write_designs(df, MODEL)
dc.write_fasta(df, MODEL)

# Quick faithfulness readout: per-protein WT sequence recovery distribution
import numpy as np
rec = df.apply(lambda r: sum(a==b for a,b in zip(r.designed_sequence, r.wt_sequence))/r.seq_length, axis=1)
print(f"\nSeq-recovery vs WT — median {rec.median():.1%}, "
      f"IQR [{rec.quantile(.25):.1%}, {rec.quantile(.75):.1%}]")
print("(Inverse-folding designs typically recover ~30-55% of WT; "
      "near-100% means the sampler is too cold / stuck, near-5% means random.)")

import os, shutil
_safe = MODEL.replace("/", "_")
_csv = str(dc.OUTPUT_DIR / f"designs_{_safe}.csv")
print("[saved locally]", _csv)
# persist to Drive if mounted (survives runtime resets)
for _d in ["/content/drive/MyDrive/decoding_bias_designs"]:
    try:
        if os.path.isdir("/content/drive/MyDrive"):
            os.makedirs(_d, exist_ok=True); shutil.copy(_csv, _d)
            print("[saved to Drive]", _d)
    except Exception as _e:
        print("(Drive copy skipped:", _e, ")")
try:
    from google.colab import files; files.download(_csv)
except Exception as _e:
    print("(browser download skipped:", _e, ") — grab it from the file browser at", _csv)